## Context Aware & Stateful Tools

In [1]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from pydantic import BaseModel
from langchain.tools import tool, ToolRuntime
from langgraph.store.memory import InMemoryStore

load_dotenv()

True

In [2]:

class UserContext(BaseModel):
    """User-specific context passed at runtime."""
    user_id: str
    user_name: str

In [3]:
@tool
def add_milestone(task: str, runtime: ToolRuntime[UserContext]) -> str:
    """Add a new todo task for the current user."""
    writer = runtime.stream_writer
    user_id = runtime.context.user_id
    user_name = runtime.context.user_name

    # Load existing milestones (if any)
    milestones = []
    if runtime.store:
        stored = runtime.store.get(("milestones",), user_id)
        if stored:
            milestones = stored.value.get("tasks", [])
            writer(f"Loaded {len(milestones)} existing milestones for {user_name}.")

    # Create new milestone
    new_milestone = {
        "id": len(milestones) + 1,
        "task": task,
        "completed": False,
    }
    milestones.append(new_milestone)

    # Persist updated list
    if runtime.store:
        runtime.store.put(("milestones",), user_id, {"tasks": milestones})
        writer(f"Saved new milestone '{task}' for {user_name}.")

    return f"Added milestone for {user_name}: '{task}' (ID: {new_milestone['id']})"


In [4]:
@tool
def check_milestones(runtime: ToolRuntime[UserContext]) -> str:
    """View all milestones for the current user."""
    writer = runtime.stream_writer
    user_id = runtime.context.user_id
    user_name = runtime.context.user_name

    milestones = []
    if runtime.store:
        stored = runtime.store.get(("milestones",), user_id)
        if stored:
            milestones = stored.value.get("tasks", [])

    if not milestones:
        return f"{user_name}, you have no milestones yet!"

    writer(f"Generating milestone list for {user_name}.")
    result_lines = [f"{user_name}'s Milestone List:"]
    for m in milestones:
        status = "Done" if m["completed"] else "Pending"
        result_lines.append(f"{status} {m['id']}. {m['task']}")

    return "\n".join(result_lines)


In [5]:
@tool
def complete_milestone(milestone_id: int, runtime: ToolRuntime[UserContext]) -> str:
    """Mark a milestone as completed for the current user."""
    writer = runtime.stream_writer
    user_id = runtime.context.user_id
    user_name = runtime.context.user_name

    # Load milestones
    milestones = []
    if runtime.store:
        stored = runtime.store.get(("milestones",), user_id)
        if stored:
            milestones = stored.value.get("tasks", [])

    # No milestones at all
    if not milestones:
        return f"{user_name}, you have no milestones yet!"

    # Find milestone
    for m in milestones:
        if m["id"] == milestone_id:
            writer(f"Marking milestone '{m['task']}' as completed for {user_name}.")
            m["completed"] = True

            # Save updated list
            if runtime.store:
                runtime.store.put(("milestones",), user_id, {"tasks": milestones})

            return f"Marked '{m['task']}' as completed for {user_name}!"

    # Not found
    writer(f"Milestone with ID {milestone_id} not found for {user_name}.")
    return f"Milestone with ID {milestone_id} not found."


In [6]:
@tool
def delete_milestone(milestone_id: int, runtime: ToolRuntime[UserContext]) -> str:
    """Delete a milestone for the current user."""
    writer = runtime.stream_writer
    user_id = runtime.context.user_id
    user_name = runtime.context.user_name

    # Load milestones
    milestones = []
    if runtime.store:
        stored = runtime.store.get(("milestones",), user_id)
        if stored:
            milestones = stored.value.get("tasks", [])

    # No milestones at all
    if not milestones:
        return f"{user_name}, you have no milestones yet!"

    # Remove milestone
    original_len = len(milestones)
    updated = [m for m in milestones if m["id"] != milestone_id]

    # If deletion happened
    if len(updated) < original_len:
        writer(f"Deleted milestone {milestone_id} for {user_name}.")
        if runtime.store:
            runtime.store.put(("milestones",), user_id, {"tasks": updated})
        return f"Deleted milestone {milestone_id} for {user_name}!"

    # Not found
    writer(f"Milestone with ID {milestone_id} not found for {user_name}.")
    return f"Milestone with ID {milestone_id} not found."

In [ ]:

store = InMemoryStore()

agent = create_agent(
    model='openai:gpt-5-nano',
    tools=[add_milestone, check_milestones, complete_milestone, delete_milestone],
    system_prompt="""You are a helpful todo assistant. 
        Help users manage their todo lists using the available tools.
        Always be friendly and confirm actions.""",
    context_schema=UserContext, 
    store=store
)

#### Lets add 3 Tasks in our milestone.
1. Building Lakehouse in databricks
2. Data governance in databricks
3. Lakeflow Jobs in databricks

In [8]:
response = agent.invoke(
        {"messages": [HumanMessage(content="Include in my milestone: Complete databricks demo for building Lakehouse")]},
        context=UserContext(user_id="user_1", user_name="Shivam")
    )
print(f"Milestone Assistant: {response['messages'][-1].content}\n")

Milestone Assistant: All set! I added the milestone:
- ID 1: Complete databricks demo for building Lakehouse

Would you like to add a due date, priority, or additional tasks? I can also list all your milestones or mark this one as in progress/completed later.



In [9]:
response = agent.invoke(
        {"messages": [HumanMessage(content="Include in my milestone: Complete databricks demo for data governance")]},
        context=UserContext(user_id="user_1", user_name="Shivam")
    )
print(f"Milestone Assistant: {response['messages'][-1].content}\n")

Milestone Assistant: All set! I added the milestone: "Complete databricks demo for data governance" (ID: 2).

Would you like to:
- View all milestones
- Mark this milestone as completed when you finish
- Add a due date, reminder, or priority
- Edit or delete this milestone?



In [10]:
response = agent.invoke(
        {"messages": [HumanMessage(content="Include in my milestone: Complete databricks demo for Lakeflow jobs")]},
        context=UserContext(user_id="user_1", user_name="Shivam")
    )
print(f"Milestone Assistant: {response['messages'][-1].content}\n")

Milestone Assistant: All set! I added the milestone: "Complete databricks demo for Lakeflow jobs" (ID: 3) for Shivam.

Would you like to:
- set a due date or priority,
- add more milestones,
- view all milestones, or
- mark any milestone as completed?



**Look at how our `InMemoryStore` looks like.**

In [11]:
items = store.search(("milestones",))
for item in items:
    for milestone in item.value['tasks']:
        print(f"ID: {milestone['id']}, Task: {milestone['task']}, Completed: {milestone['completed']}")

ID: 1, Task: Complete databricks demo for building Lakehouse, Completed: False
ID: 2, Task: Complete databricks demo for data governance, Completed: False
ID: 3, Task: Complete databricks demo for Lakeflow jobs, Completed: False


### Lets ask our agent give us the tasks list

In [12]:
response = agent.invoke(
        {"messages": [HumanMessage(content="Show me my milestones and how many are completed")]},
        context=UserContext(user_id="user_1", user_name="Shivam")
    )
print(f"Milestone Assistant: {response['messages'][-1].content}\n")

Milestone Assistant: Here’s your milestone status:

- Total milestones: 3
- Completed: 0
- Pending:
  1) Complete databricks demo for building Lakehouse
  2) Complete databricks demo for data governance
  3) Complete databricks demo for Lakeflow jobs

Would you like me to mark any as completed, add a new milestone, or delete one? If you want to mark, tell me the milestone number (e.g., "mark 2 as completed").



#### Tell agent that we have completed Databricks Data Governance
Agent should autmatically update in milestone and mark this task as completed.

In [13]:
response = agent.invoke(
        {"messages": [HumanMessage(content="I completed databricks data governance demo, please update my milestone")]},
        context=UserContext(user_id="user_1", user_name="Shivam")
    )
print(f"Milestone Assistant: {response['messages'][-1].content}\n")

Milestone Assistant: Great job on finishing the data governance demo! I’ve marked the milestone as completed.

Here’s your updated milestone list:
- Pending 1: Complete databricks demo for building Lakehouse
- Pending 3: Complete databricks demo for Lakeflow jobs

Would you like me to mark the remaining milestones as completed as well, or add any new tasks?



### Lets re-check how many tasks are pending or completed now

In [14]:
response = agent.invoke(
        {"messages": [HumanMessage(content="Show me my milestones and how many are completed")]},
        context=UserContext(user_id="user_1", user_name="Shivam")
    )
print(f"Milestone Assistant: {response['messages'][-1].content}\n")

Milestone Assistant: Here you go! Here’s the status of your milestones:

Milestones:
- Pending: Complete databricks demo for building Lakehouse
- Done: Complete databricks demo for data governance
- Pending: Complete databricks demo for Lakeflow jobs

Overview:
- Total milestones: 3
- Completed: 1
- Pending: 2

Would you like me to mark any of these as completed, add a new milestone, or delete one?



#### Lets try to delete all tasks from our milestone.
Here model is responding for further conformation before making the delete action.

In [19]:
response = agent.invoke(
        {"messages": [HumanMessage(content="Please delete all tasks from my milestones")]},
        context=UserContext(user_id="user_1", user_name="Shivam")
    )
print(f"Milestone Assistant: {response['messages'][-1].content}\n")

Milestone Assistant: I can help with that, but I want to confirm since this will permanently delete all milestones (tasks) for your account.

This action cannot be undone. Do you want me to proceed and delete all milestones now? 
If you'd like, I can first show you the list of milestones that will be deleted (IDs and titles) and then proceed after your confirmation.



### Lets try to fetch tasks for a new user.

In [21]:
response = agent.invoke(
        {"messages": [HumanMessage(content="Show me my milestones and how many are completed")]},
        context=UserContext(user_id="user_5", user_name="Satyam")
    )
print(f"Milestone Assistant: {response['messages'][-1].content}\n")

Milestone Assistant: Here’s what I found:

- Milestones: none yet
- Completed: 0
- Total milestones: 0

Would you like to add your first milestone? If you tell me the task, I’ll add it right away (e.g., “Finish project proposal” or “Pay electricity bill”).

